In [1]:
import pandas as pd
import psycopg2

sheet_curr = '1wxR3iYna86qrdViwHjUPzHuw6bCNeMLb72M25hpUHYk'
sheet_archive = '1PxNYGMXaVrRqI0uyMQF46K7nDEG16WnDoKrFyI_qrvE'
gid_matches = '2141931777'
gid_deck = '590005429'

# MATCH_ID = 1000000000
# EVENT_ID = 1000000
# DECK_ID  = 1000
# EVENT_TYPE_ID = 100

In [2]:
with open("credentials.txt", "r") as file:
    credentials = [line.strip() for line in file]

In [3]:
def conn(query,vars=()):
    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        if len(vars) > 0:
            cursor.execute(query,vars)
        else:
            cursor.execute(query)

        conn.commit()
    except psycopg2.Error as e:
        print('Error:', e)
    finally:
        if conn:
            cursor.close()
            conn.close()

In [4]:
def parse_class_sheet(sheet,gid,deck_id_start=1000,event_type_id_start=100):
    deck_url = f'https://docs.google.com/spreadsheets/d/{sheet}/export?format=csv&gid={gid}'
    df = pd.read_csv(deck_url)

    # Create dataframe with valid Deck Names.
    df_decks = df[['Archetype','Subarchetype']].sort_values(['Archetype','Subarchetype']).reset_index(drop=True)
    df_decks.columns = ['ARCHETYPE','SUBARCHETYPE']
    df_decks['ARCHETYPE'] = df_decks['ARCHETYPE'].str.strip().str.upper()
    df_decks['SUBARCHETYPE'] = df_decks['SUBARCHETYPE'].str.strip().str.upper()

    # Adding Deck IDs.
    count = deck_id_start
    df_decks['DECK_ID'] = 0
    for index, row in df_decks.iterrows():
        df_decks.at[index,'DECK_ID'] = count
        count += 1

    df_decks = pd.concat([df_decks,pd.DataFrame({'ARCHETYPE':['NA','NA'],'SUBARCHETYPE':['NA','NO SHOW'],'DECK_ID':[1998,1999]})],ignore_index=True)

    # Create dataframe with valid Event Types.
    df_events = df[['Event Types']]
    df_events.columns = ['EVENT_TYPE']
    df_events = df_events.dropna(subset=['EVENT_TYPE'])
    df_events['EVENT_TYPE'] = df_events['EVENT_TYPE'].str.strip().str.upper()

    # Adding Event Type IDs.
    count = event_type_id_start
    df_events['EVENT_TYPE_ID'] = 0
    for index, row in df_events.iterrows():
        df_events.at[index,'EVENT_TYPE_ID'] = count
        count += 1

    # Add Format column.
    df_decks['FORMAT'] = 'VINTAGE'
    df_decks = df_decks[['FORMAT','ARCHETYPE','SUBARCHETYPE','DECK_ID']]

    df_events['FORMAT'] = 'VINTAGE'
    df_events = df_events[['FORMAT','EVENT_TYPE','EVENT_TYPE_ID']]
    
    return (df_decks,df_events)

In [6]:
def insert(df_valid_decks=None, df_valid_event_types=None, credentials=credentials):
    valid_decks_query = """
        INSERT INTO VALID_DECKS (FORMAT, ARCHETYPE, SUBARCHETYPE, DECK_ID)
        VALUES (%s, %s, %s, %s)
    """
    valid_event_types_query = """
        INSERT INTO VALID_EVENT_TYPES (FORMAT, EVENT_TYPE, EVENT_TYPE_ID)
        VALUES (%s, %s, %s)
    """

    try:
        conn = psycopg2.connect(
            host=credentials[0],
            port=credentials[1],
            user=credentials[2],
            password=credentials[3],
            database=credentials[4]
        )
        cursor = conn.cursor()

        # Insert valid_decks
        if df_valid_decks is not None:
            values_list = [(row['FORMAT'], row['ARCHETYPE'], row['SUBARCHETYPE'], row['DECK_ID']) for index, row in df_valid_decks.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_decks_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_DECKS: {values} | Error: {e}")
                    continue  # Skip the row and continue with the next one
            conn.commit()

        # Insert valid_event_types
        if df_valid_event_types is not None:
            values_list = [(row['FORMAT'], row['EVENT_TYPE'], row['EVENT_TYPE_ID']) for index, row in df_valid_event_types.iterrows()]
            for values in values_list:
                try:
                    print(values)
                    cursor.execute(valid_event_types_query, values)
                except Exception as e:
                    print(f"Error inserting row into VALID_EVENT_TYPES: {values} | Error: {e}")
                    continue
            conn.commit()

    except Exception as e:
        print(f"Error occurred while loading data: {e}")
        conn.rollback()

    finally:
        if conn:
            cursor.close()
            conn.close()

In [7]:
df_valid_decks, df_valid_event_types = parse_class_sheet(sheet_curr,gid_deck)

In [9]:
insert(df_valid_decks,df_valid_event_types)

('VINTAGE', 'AGGRO', 'ELDRAZI', 1000)
Error inserting row into VALID_DECKS: ('VINTAGE', 'AGGRO', 'ELDRAZI', 1000) | Error: duplicate key value violates unique constraint "valid_decks_pkey"
DETAIL:  Key (deck_id)=(1000) already exists.

('VINTAGE', 'AGGRO', 'INITIATIVE', 1001)
Error inserting row into VALID_DECKS: ('VINTAGE', 'AGGRO', 'INITIATIVE', 1001) | Error: current transaction is aborted, commands ignored until end of transaction block

('VINTAGE', 'AGGRO', 'MERFOLK', 1002)
Error inserting row into VALID_DECKS: ('VINTAGE', 'AGGRO', 'MERFOLK', 1002) | Error: current transaction is aborted, commands ignored until end of transaction block

('VINTAGE', 'AGGRO', 'OTHER AGGRO', 1003)
Error inserting row into VALID_DECKS: ('VINTAGE', 'AGGRO', 'OTHER AGGRO', 1003) | Error: current transaction is aborted, commands ignored until end of transaction block

('VINTAGE', 'AGGRO', 'RED PRISON', 1004)
Error inserting row into VALID_DECKS: ('VINTAGE', 'AGGRO', 'RED PRISON', 1004) | Error: current t